# Train scProto — affinity comparison (arbf / mean product / BANKSY)

Trains the full scProto pipeline (CVAE pretrain + prototype-UMAP) on **s28nsc**
once per affinity graph:

| Affinity | `affinity_type` | What it encodes |
|---|---|---|
| PCA only (baseline) | `arbf` | Transcriptomics only, adaptive RBF (SEACells kernel) |
| Mean product | `mean_product` | `rbf(own PCA) x rbf(mean-neighbour PCA)` — soft AND logic, k=35 spatial context |
| BANKSY | `banksy0.5` | `concat(own PCA, mean-neighbour PCA)` -> single RBF, alpha=0.5 |

Each run trains its own encoder + prototypes end-to-end and saves checkpoints
(pretrain checkpoint, UMAP checkpoint, metacells, metrics) under
`MODEL_DIR/s28nsc/<model_name>/` — see the **Reload elsewhere** section at the
bottom for how another notebook picks these back up for metric comparison
without retraining.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q scarches SEACells faiss-gpu-cu12 scib-metrics

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 135.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.1/699.1 kB 56.0 MB/

In [ ]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


## Config

Same dataset for all three runs so the resulting models are directly
comparable. Hyperparameters mirror the established `s28nsc` recipe from
`train_scproto.ipynb` (`LAMBDA_PROTO_UMAP_PRECON` + `nassoc_agg='max'`).

In [ ]:
from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.spatial_immune_task import NSCLC_EVAL_GROUPS

DS_ID = 's28nsc'

COMMON_KWARGS = dict(
    cvae_epochs=50,
    train_epochs=50,
    eval_freq=3,
    patience=6,
    batch_size=1024,
    umap_steps_per_epoch=500,
    niche_key='niches_3D',
    target_groups=NSCLC_EVAL_GROUPS,
    lambda_config=LAMBDA_PROTO_UMAP_PRECON | {'nassoc_agg': 'max'},
)

# Set True on a re-run to skip training and just reload each saved UMAP checkpoint.
LOAD_UMAP = False

AFFINITIES = ['arbf', 'mean_product', 'banksy0.5']

trainers = {}
results = {}
mc_adatas = {}

## Train — one cell per affinity

Each cell is independent — skip/re-run any one without affecting the others.

### arbf (PCA-only baseline)

In [ ]:
t, res, mc_ad = run_mc_task(DS_ID, affinity_type='arbf', load_umap=LOAD_UMAP, **COMMON_KWARGS)
trainers['arbf'], results['arbf'], mc_adatas['arbf'] = t, res, mc_ad
print(res)

 captum (see https://github.com/pytorch/captum).


dataset is None, loading s28nsc
loading s28nsc data
⚠️ No HVG column found.
proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

📊 Affinity: wdeg[min/mean/max]=1.315/25.499/124.874, effk_med=61.6, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'section'}
[waypoint init] N=58423  K=800  n_eigs=10  nnz=4352008  nnz/row=74.5  w[min/mean/max]=3.114e-03/3.423e-01/9.835e-01  deg[min/mean/max]=1.31/25.50/124.87
[w

waypoint MaxMin: 100%|██████████| 799/799 [00:00<00:00, 839.94proto/s]


[waypoint init] selected 800 seed cells


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/58 [00:00<?, ?it/s]

Saved metacells (800 prototypes) to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31/metacells.h5ad


  0%|          | 0/49 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

UMAP data saved to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
{'purity': 0.8619394497269813, 'niche_purity': 0.8559889901645817, 'batch_entropy': -1.0000000826903712e-10, 'modularity': 0.46529812761429373, 'coverage': 1.0, 'dge_rbo_avg': 0.17448630265777365, 'dge_kendall_avg': 0.05627731316780108, 'dge_jaccard_avg': 0.09334003857560122, 'scgraph_corr_avg': 0.353693482871946, 'ct_niche_rbo_avg': 0.030280002541031206, 'aff_compactness_per_batch': {'section_28': 0.21207960536936926}, 'aff_compactness_mean': 0.13513359875958822, 'tumor_cells_tumor_surface_purity': 0.34455557527945607, 'tumor_cells_tumor_surface_coverage': 0.5686202461877641, 'tumor_cells_tumor_surface_homogeneity': 0.5632923020393166, 'tumor_cells_tumor_core_purity': 0.33486030630110225, 'tumor_cells_tumor_core_coverage': 0.3832035595105673, 'tumor_cells_tumor_core_homogeneity': 0.6031701890989989, 'macrop

### mean_product

In [ ]:
t, res, mc_ad = run_mc_task(DS_ID, affinity_type='mean_product', load_umap=LOAD_UMAP, **COMMON_KWARGS)
trainers['mean_product'], results['mean_product'], mc_adatas['mean_product'] = t, res, mc_ad
print(res)

dataset is None, loading s28nsc
loading s28nsc data
⚠️ No HVG column found.
proto_umap_ds-s28n_NP800_prtInit-wayp_aff-mean_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

[3609] Generating affinities..., saving data to: ./graphs/affinity_s28nsc58423_ncomp50_kneighbors50_mean_product.pkl_tmp.h5ad
📊 Affinity: wdeg[min/mean/max]=0.021/2.334/7.080, effk_med=39.1, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'section'}
[waypoint init] N=58423  K=800  

waypoint MaxMin: 100%|██████████| 799/799 [00:00<00:00, 855.00proto/s]


[waypoint init] selected 800 seed cells


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

...

  0%|          | 0/58 [00:00<?, ?it/s]

Saved metacells (800 prototypes) to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-mean_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31/metacells.h5ad


  0%|          | 0/49 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

UMAP data saved to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-mean_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
{'purity': 0.8467209709923746, 'niche_purity': 0.7792931579811618, 'batch_entropy': -1.000000082690371e-10, 'modularity': 0.5979053654516641, 'coverage': 1.0, 'dge_rbo_avg': 0.15327654516674694, 'dge_kendall_avg': 0.16067633809892518, 'dge_jaccard_avg': 0.08908304466555325, 'scgraph_corr_avg': 0.19837145892638577, 'ct_niche_rbo_avg': 0.03786854164132935, 'aff_compactness_per_batch': {'section_28': 1.1659765788028096}, 'aff_compactness_mean': 0.9030584605899705, 'tumor_cells_tumor_surface_purity': 0.4390273811969707, 'tumor_cells_tumor_surface_coverage': 0.9513136138159104, 'tumor_cells_tumor_surface_homogeneity': 0.5120338048870109, 'tumor_cells_tumor_core_purity': 0.33188779872389573, 'tumor_cells_tumor_core_coverage': 0.005839822024471635, 'tumor_cells_tumor_core_homogeneity': 0.6154060066740823, 'macropha

### BANKSY (alpha=0.5)

In [ ]:
t, res, mc_ad = run_mc_task(DS_ID, affinity_type='banksy0.5', load_umap=LOAD_UMAP, **COMMON_KWARGS)
trainers['banksy0.5'], results['banksy0.5'], mc_adatas['banksy0.5'] = t, res, mc_ad
print(res)

dataset is None, loading s28nsc
loading s28nsc data
⚠️ No HVG column found.
proto_umap_ds-s28n_NP800_prtInit-wayp_aff-bank_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

📊 Affinity: wdeg[min/mean/max]=12.393/17.964/21.988, effk_med=49.7, mutual=50.47%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'section'}
[waypoint init] N=58423  K=800  n_eigs=10  nnz=4367898  nnz/row=74.8  w[min/mean/max]=7.470e-02/2.403e-01/8.623e-01  deg[min/mean/max]=6.58/17.96/111.97
[wa

waypoint MaxMin: 100%|██████████| 799/799 [00:00<00:00, 837.84proto/s]


[waypoint init] selected 800 seed cells


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

...

  0%|          | 0/58 [00:00<?, ?it/s]

Saved metacells (800 prototypes) to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-bank_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31/metacells.h5ad


  0%|          | 0/49 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

UMAP data saved to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-bank_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
{'purity': 0.9454848346328236, 'niche_purity': 0.9556897984987566, 'batch_entropy': -1.000000082690371e-10, 'modularity': 0.6323644658515475, 'coverage': 0.7222222222222222, 'dge_rbo_avg': 0.07852126083832636, 'dge_kendall_avg': 0.22059311644343244, 'dge_jaccard_avg': 0.07217606968953304, 'scgraph_corr_avg': 0.7911918682131401, 'ct_niche_rbo_avg': 0.013432640301495075, 'aff_compactness_per_batch': {'section_28': 16.082431193375523}, 'aff_compactness_mean': 10.32309958344581, 'tumor_cells_tumor_surface_purity': 0.4524932324447214, 'tumor_cells_tumor_surface_coverage': 0.9318390593422745, 'tumor_cells_tumor_surface_homogeneity': 0.3825096454161308, 'tumor_cells_tumor_core_purity': 0.32731270601488965, 'tumor_cells_tumor_core_coverage': 0.002502780867630701, 'tumor_cells_tumor_core_homogeneity': 0.4332591768631

## Compare metrics across affinities

In [ ]:
import pandas as pd

metrics_df = pd.DataFrame(results).T
metrics_df

,purity,niche_purity,batch_entropy,modularity,coverage,dge_rbo_avg,dge_kendall_avg,dge_jaccard_avg,scgraph_corr_avg,ct_niche_rbo_avg,...,fibroblasts_vascular_stroma_coverage,fibroblasts_vascular_stroma_homogeneity,tumor_cells_macrophage_islands_purity,tumor_cells_macrophage_islands_coverage,tumor_cells_macrophage_islands_homogeneity,tumor_tumor_core_celltype_purity,tumor_tumor_core_niche_purity,tumor_tumor_surface_celltype_purity,tumor_tumor_surface_niche_purity,tumor_pseudotime_spread_mean
arbf,0.861939,0.855989,-0.0,0.465298,1.0,0.174486,0.056277,0.09334,0.353693,0.03028,...,0.00413,0.342359,0.031912,0.008651,0.34083,0.8029,0.3349,0.6775,0.3446,0.1876
mean_product,0.846721,0.779293,-0.0,0.597905,1.0,0.153277,0.160676,0.089083,0.198371,0.037869,...,0.004589,0.372648,0.047418,0.008651,0.477509,0.9224,0.3319,0.8759,0.439,0.1456
banksy0.5,0.945485,0.95569,-0.0,0.632364,0.722222,0.078521,0.220593,0.072176,0.791192,0.013433,...,0.002295,0.374943,0.042511,0.00173,0.297578,0.953,0.3273,0.9187,0.4525,0.1614


## Visualize — UMAP per affinity

Colored by cell type and (3D) niche, prototypes overlaid.

In [ ]:
for name, t in trainers.items():
    print(f'--- {name} ---')
    fig, proto_labels = t.plot_umap_simple(
        color_key=['celltypes', 'niches_3D'],
        show_proto_nums=False,
    )

## Reload elsewhere — for metric comparison in other notebooks

Each run's artifacts live under `MODEL_DIR/s28nsc/<model_name>/`
(`umap_checkpoint.pth`, `metacells.h5ad`, `metrics.json`, `clusters.npz`, ...).
The model name is derived from the hyperparameters passed to `get_trainer` /
`run_mc_task`, so **another notebook reloads a specific run by calling
`run_mc_task` with the exact same arguments plus `load_umap=True`** — this
skips training entirely and just loads the saved checkpoint:

```python
from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.spatial_immune_task import NSCLC_EVAL_GROUPS

COMMON_KWARGS = dict(
    cvae_epochs=50, train_epochs=50, eval_freq=3, patience=6,
    batch_size=1024, umap_steps_per_epoch=500,
    niche_key='niches_3D', target_groups=NSCLC_EVAL_GROUPS,
    lambda_config=LAMBDA_PROTO_UMAP_PRECON | {'nassoc_agg': 'max'},
)

t_arbf, res_arbf, mc_arbf = run_mc_task(
    's28nsc', affinity_type='arbf', load_umap=True, **COMMON_KWARGS
)
# same for 'mean_product' and 'banksy0.5'
```

`mc_ad` / `mc_adatas[...]` is the metacell-level AnnData (K prototypes x genes);
`t.train_ds.adata.obs['metacell_id']` has the per-cell prototype assignment.
Both are also saved to disk (`metacells.h5ad`) so they can be loaded with
`sc.read_h5ad(...)` directly if a fresh trainer isn't needed.